In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy




TO_A_ROUTE = f"{os.getcwd()}/../data/E01_toA/NEW"
EXCLUDED_TO_A = ["RESULTS", "E01_toA.zip"]

TIMESLOTS = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


TO_B_ROUTE = f"{os.getcwd()}/../data/E01_toB"
EXCLUDED_TO_B = ["RESULTS", "E01_toB.zip"]

In [2]:
final_df_toA = pd.DataFrame

final_df_toB = pd.DataFrame


dataframes_toA = []

# TOA
for csv_file in os.listdir(TO_A_ROUTE):
    if csv_file.endswith('.csv') and csv_file not in EXCLUDED_TO_A:
        file_path = os.path.join(TO_A_ROUTE, csv_file)
        
        auxiliar = pd.read_csv(file_path, encoding="utf-8", header=0)
        
        dataframes_toA.append(auxiliar)

if dataframes_toA:
    final_df_toA = pd.concat(dataframes_toA, ignore_index=True)
else:
    final_df_toA = pd.DataFrame()


STOPS_TOA = final_df_toA["Stop"].unique()


dataframes_toB = []

# TOA
for csv_file in os.listdir(TO_B_ROUTE):
    if csv_file.endswith('.csv') and csv_file not in EXCLUDED_TO_B:
        file_path = os.path.join(TO_B_ROUTE, csv_file)
        
        auxiliar = pd.read_csv(file_path, encoding="utf-8", header=0)
        
        dataframes_toB.append(auxiliar)

if dataframes_toB:
    final_df_toB = pd.concat(dataframes_toB, ignore_index=True)
else:
    final_df_toB = pd.DataFrame()


STOPS_TOB = final_df_toB["Stop"].unique()

In [3]:
def clean_dataframe(dataframe, daytype, direction):
    dataframe = dataframe[dataframe["DayType"] == daytype]
    dataframe = dataframe[dataframe["Direction"] == direction]
    dataframe = dataframe[dataframe["Occupation"] <= 80]
    dataframe = dataframe[dataframe["Occupation"] >= 0]
    dataframe = dataframe[dataframe["Board"] <= 80]
    dataframe = dataframe[dataframe["Board"] >= 0]
    dataframe = dataframe[dataframe["DropOff"] <= 80]
    dataframe = dataframe[dataframe["DropOff"] >= 0]
    dataframe = dataframe[dataframe["FreeDay"] == 0]

    return dataframe

In [4]:
final_df_toA = clean_dataframe(final_df_toA, "LA", 2)
final_df_toB = clean_dataframe(final_df_toB, "LA", 1)

In [6]:
#KPI SAVER PER STOP. ONLY GENERAL METRICS


kpi_toA = {}
kpi_toB = {}
#Ocupación media de los buses

kpi_toA["Mean_occupancy"] = (final_df_toA["Occupation"].mean()/80)*100

kpi_toB["Mean_occupancy"] = (final_df_toB["Occupation"].mean()/80)*100

#Media de buses fisicos encargados

each_day_unique_buses = []
each_day_total_trips_toA = []

for unique_day in final_df_toA["EventDate"].unique():
    that_day_data = final_df_toA[final_df_toA["EventDate"] == unique_day]
    each_day_total_trips_toA.append(len(that_day_data["TripId"].unique()))
    each_day_unique_buses.append(len(that_day_data["LogicVehicle"].unique()))

kpi_toA["Mean_unique_buses"] = np.mean(each_day_unique_buses)

each_day_unique_buses = []

each_day_total_trips_toB = []

for unique_day in final_df_toB["EventDate"].unique():
    that_day_data = final_df_toB[final_df_toB["EventDate"] == unique_day]
    each_day_total_trips_toB.append(len(that_day_data["TripId"].unique()))
    each_day_unique_buses.append(len(that_day_data["LogicVehicle"].unique()))

kpi_toB["Mean_unique_buses"] = np.mean(each_day_unique_buses)


kpi_toA["Mean_total_trips"] = np.mean(each_day_total_trips_toA) + np.mean(each_day_total_trips_toB)
kpi_toB["Mean_total_trips"] = np.mean(each_day_total_trips_toA) + np.mean(each_day_total_trips_toB)


kpi_toA, kpi_toB

({'Mean_occupancy': np.float64(33.93928940759344),
  'Mean_unique_buses': np.float64(11.455696202531646),
  'Mean_total_trips': np.float64(163.14556962025318)},
 {'Mean_occupancy': np.float64(30.038444557522787),
  'Mean_unique_buses': np.float64(11.474683544303797),
  'Mean_total_trips': np.float64(163.14556962025318)})